# MicroSplit demo with the CAREamist API: HT-LIF24, 5 ms, 2 channels


## 1. Imports


In [ ]:
from pathlib import Path

import lightning.pytorch as L
import matplotlib.pyplot as plt
import numpy as np
import tifffile

from careamics import CAREamist
from careamics.config import (
    create_advanced_microsplit_config,
    create_n2v_config,
)
from careamics.lvae_training.dataset.utils.data_utils import get_datasplit_tuples
from careamics.lvae_training.metrics import compute_stats
from careamics.models.lvae.noise_models import MultiChannelNoiseModel
from careamics.noise_model import NoiseModelTrainer

## 2. Parameters

In [ ]:
DATA_ROOT = Path("/group/jug/public_html/microsplit/ht_lif24_tiff")
EXPOSURE = "5ms"            # noisy input and training data
HIGHSNR_EXPOSURE = "500ms"  # ground truth for the evaluation
CH_IDX_LIST = [0, 1, 8]     # targets [Nucleus, MicroTubules] + superimposed input "01"
N_TARGETS = len(CH_IDX_LIST) - 1

WORK_DIR = Path("microsplit_lif24_5ms_careamist_demo").resolve()
EXPERIMENT_NAME = "ht_lif24_5ms_careamist_demo"
SEED = 42

# Data split (same fractions as the paper)
VAL_FRACTION = 0.1
TEST_FRACTION = 0.1

# Noise model
NM_PATHS = None  # e.g. [WORK_DIR / "noise_models" / "noise_model_Ch0.npz", ...]
N2V_NUM_EPOCHS = 10
NM_N_GAUSSIAN = 6
NM_N_COEFF = 4
NM_MIN_SIGMA = 0.125  # clamps the VARIANCE, not sigma. Do not use 200 on this data.
NM_N_EPOCHS = 2000

# MicroSplit
PATCH_SIZE = (64, 64)
GRID_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = 4
NUM_EPOCHS = 10
Z_DIMS = [128] * 4
N_FILTERS = 64
MULTISCALE_COUNT = 3
NOISE_MODEL_LIKELIHOOD_WEIGHT = 0.9
GAUSSIAN_LIKELIHOOD_WEIGHT = 0.1

# Prediction
MMSE_COUNT = 50   # posterior samples to average (at least 2); the paper uses 50
EVAL_SPLIT = "test"  # "test" = the 10 paper test frames, "val" = the 12 validation frames

# Pretrained model (our paper reproduction, 400 epochs)
USE_PRETRAINED = True
PRETRAINED_DIR = Path("/home/igor.zubarev/projects/careamics/scripts")
PRETRAINED_CKPT = (
    PRETRAINED_DIR / "lvae_checkpoints/5ms/ht_lif24_5ms_ngds_v3_400ep/checkpoints/last.ckpt"
)
# The noise models of the same run. Prediction does not use them: the noise
# model is only in the loss. Section 4.3 loads them.
PRETRAINED_NM_PATHS = [
    PRETRAINED_DIR / f"noise_models/5ms_ngds_ht_lif24_5ms_ngds_v3_400ep/noise_model_ch{c}.npz"
    for c in range(2)
]
if USE_PRETRAINED:
    NM_PATHS = PRETRAINED_NM_PATHS

WORK_DIR.mkdir(parents=True, exist_ok=True)
L.seed_everything(SEED, workers=True)

## 3. Load the data

I dunno this is still a bit ugly :(

In [ ]:
def load_exposure(exposure: str) -> np.ndarray:
    # Return all frames of Set1..Set6 as float32 SCYX, channels = CH_IDX_LIST.
    stacks = [
        tifffile.imread(DATA_ROOT / exposure / f"Set{i}" / f"uSplit_{exposure}.tif")[
            :, CH_IDX_LIST
        ]
        for i in range(1, 7)
    ]
    return np.concatenate(stacks, axis=0).astype(np.float32)


data = load_exposure(EXPOSURE)
train_idx, val_idx, test_idx = get_datasplit_tuples(
    VAL_FRACTION, TEST_FRACTION, len(data)
)
print(f"{EXPOSURE} data: {data.shape}")
print(f"train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)} frames")

# The network input is the last channel; the targets are the other channels.
train_input, train_target = data[train_idx, -1:], data[train_idx, :-1]
val_input, val_target = data[val_idx, -1:], data[val_idx, :-1]

Show one training frame. The first panel is the network input. The other panels are the two target channels at 5 ms.

In [ ]:
frame = 0
titles = ["Input (superimposed)", "Target 0: Nucleus", "Target 1: MicroTubules"]
images = [train_input[frame, 0], train_target[frame, 0], train_target[frame, 1]]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap="magma")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## 4. Noise model training

### 4.1 Train N2V


In [ ]:
nm_dir = WORK_DIR / "noise_models"

if NM_PATHS is None:
    nm_observation = np.moveaxis(data[train_idx], 1, -1)  # SCYX -> SYXC

    n2v_config = create_n2v_config(
        experiment_name=f"{EXPERIMENT_NAME}_n2v",
        data_type="array",
        axes="SYXC",
        n_channels=len(CH_IDX_LIST),
        patch_size=PATCH_SIZE,
        batch_size=BATCH_SIZE,
        num_epochs=N2V_NUM_EPOCHS,
    )
    n2v_config.data_config.seed = SEED
    n2v = CAREamist(config=n2v_config, work_dir=str(nm_dir))
    n2v.train(train_data=nm_observation)

    n2v_prediction, _ = n2v.predict(nm_observation, tile_size=(256, 256))
    nm_signal = np.concatenate(n2v_prediction, axis=0)  # SYXC
    print(f"N2V signal estimate: {nm_signal.shape}")

Compare the noisy data with the N2V signal estimate for one target channel. The signal estimate must show the same structures with less noise.

In [ ]:
if NM_PATHS is None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(nm_observation[frame, ..., 1], cmap="magma")
    axes[0].set_title("Noisy 5 ms, MicroTubules")
    axes[1].imshow(nm_signal[frame, ..., 1], cmap="magma")
    axes[1].set_title("N2V signal estimate")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()

### 4.2 Fit the GMM noise models

In [ ]:
if NM_PATHS is None:
    nm_trainer = NoiseModelTrainer(
        n_gaussian=NM_N_GAUSSIAN,
        n_coeff=NM_N_COEFF,
        min_sigma=NM_MIN_SIGMA,
    )
    nm_trainer.train_from_pairs(
        signal=np.clip(nm_signal[..., :N_TARGETS], 0, None),
        observation=nm_observation[..., :N_TARGETS],
        axes="SYXC",
        n_epochs=NM_N_EPOCHS,
    )
    NM_PATHS = nm_trainer.save(nm_dir)
    print(f"Saved noise models: {NM_PATHS}")

Show the fit loss of each GMM. The loss must decrease and then become flat. If the loss is `nan`, do not use the noise model. The trainer saves the `.npz` file also when the fit fails.

In [ ]:
if "nm_trainer" in globals():
    fig, axes = plt.subplots(1, N_TARGETS, figsize=(6 * N_TARGETS, 4))
    for ch, (ax, losses) in enumerate(zip(axes, nm_trainer.train_losses)):
        ax.plot(losses)
        ax.set_title(f"GMM fit, target channel {ch}")
        ax.set_xlabel("step")
        ax.set_ylabel("negative log-likelihood")
    plt.tight_layout()

### 4.3 Load the noise models

In [ ]:
noise_model = MultiChannelNoiseModel.from_npz(NM_PATHS)
print(f"{len(noise_model)} noise models loaded")

## 5. MicroSplit training

### 5.1 Make the configuration

`create_advanced_microsplit_config` makes the full configuration: data, model, loss and trainer.

- The loss has two terms: the noise model likelihood (weight 0.9) and the Gaussian likelihood (weight 0.1).
- The model is a ladder VAE with 4 latent levels of 128 dimensions and 3 lateral contexts (`multiscale_count`).
- Mixed precision (16 bit) and gradient clipping at 0.5 are the values of the script.
- For supervised algorithms, the default configuration adds early stopping with a patience of 3 epochs. `early_stopping_params={}` disables it, so the training runs for `NUM_EPOCHS` epochs.

In [ ]:
config = create_advanced_microsplit_config(
    experiment_name=EXPERIMENT_NAME,
    data_type="array",
    axes="SCYX",
    patch_size=PATCH_SIZE,
    output_channels=N_TARGETS,
    multiscale_count=MULTISCALE_COUNT,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    gaussian_likelihood_weight=GAUSSIAN_LIKELIHOOD_WEIGHT,
    noise_model_likelihood_weight=NOISE_MODEL_LIKELIHOOD_WEIGHT,
    model_params={"z_dims": Z_DIMS, "n_filters": N_FILTERS},
    num_workers=NUM_WORKERS,
    early_stopping_params={},
    trainer_params={
        "max_epochs": NUM_EPOCHS,
        "precision": 16,
        "gradient_clip_algorithm": "value",
        "gradient_clip_val": 0.5,
    },
    seed=SEED,
)
loss = config.algorithm_config.loss
print(f"loss weights: noise model {loss.noise_model_likelihood_weight}, "
      f"gaussian {loss.gaussian_likelihood_weight}")

### 5.2 Train, or load the pretrained model

If `USE_PRETRAINED` is `False`, the cell makes a `CAREamist` from the configuration and starts the training.

- `train` gets the noise models with `noise_model=`.
- `train` calculates the normalization statistics on the training data.
- The checkpoints go to `WORK_DIR/checkpoints`. Each checkpoint also contains the configuration, the normalization statistics and the noise models.
- On one GPU, one epoch takes some minutes.

A healthy run shows some negative values of `train_loss_step`. The reconstruction log-likelihood makes these values negative. If the loss becomes `nan`, check the noise models first.

If `USE_PRETRAINED` is `True`, the cell makes a `CAREamist` from the checkpoint. The checkpoint gives the configuration, the weights and the normalization statistics. The training data is not necessary.

In [ ]:
if USE_PRETRAINED:
    careamist = CAREamist(checkpoint_path=PRETRAINED_CKPT, work_dir=WORK_DIR)
else:
    careamist = CAREamist(config=config, work_dir=WORK_DIR)
    careamist.train(
        train_data=train_input,
        train_data_target=train_target,
        val_data=val_input,
        val_data_target=val_target,
        noise_model=noise_model,
    )
print(careamist.config.data_config.normalization)

## 6. Prediction

### 6.1 Predict the evaluation frames

MicroSplit is a variational model. Each forward pass gives a different sample. The MMSE prediction is the mean of `MMSE_COUNT` samples. More samples give a better prediction, but the prediction takes more time.

`predict` cuts each frame into `64 x 64` tiles. The tiles overlap, so that only the center `GRID_SIZE x GRID_SIZE` region of each tile goes into the output. `predict` then joins the tiles into full frames.

With `n_predictions`, `predict` also gives the uncertainty: the standard deviation of the samples at each pixel. `n_predictions` must be at least 2.

After a training, `predict` uses the checkpoint with the lowest validation loss. With `USE_PRETRAINED = True`, it uses the loaded checkpoint.

In [ ]:
overlap = tuple(p - GRID_SIZE for p in PATCH_SIZE)
eval_idx = {"val": val_idx, "test": test_idx}[EVAL_SPLIT]
eval_input = data[eval_idx, -1:]

prediction, uncertainty, _ = careamist.predict(
    eval_input,
    tile_size=PATCH_SIZE,
    tile_overlap=overlap,
    n_predictions=MMSE_COUNT,
)
prediction = np.moveaxis(np.concatenate(prediction, axis=0), 1, -1)  # SCYX -> SYXC
uncertainty = np.moveaxis(np.concatenate(uncertainty, axis=0), 1, -1)
print(f"prediction: {prediction.shape}, uncertainty: {uncertainty.shape}")

### 6.2 Evaluate

In [ ]:
highsnr = load_exposure(HIGHSNR_EXPOSURE)
gt = np.moveaxis(highsnr[eval_idx, :-1], 1, -1)  # (N, Y, X, 2)

metrics = compute_stats([gt], [prediction])

### 6.3 Show the results

In [ ]:
names = ["Nucleus", "MicroTubules"]


def show(frame: int, region=(slice(None), slice(None))) -> None:
    fig, axes = plt.subplots(N_TARGETS, 3, figsize=(15, 5 * N_TARGETS))
    for ch in range(N_TARGETS):
        panels = [
            (eval_input[frame, 0], "Input (5 ms, superimposed)"),
            (prediction[frame, ..., ch], f"MicroSplit: {names[ch]}"),
            (gt[frame, ..., ch], f"500 ms GT: {names[ch]}"),
        ]
        for ax, (img, title) in zip(axes[ch], panels):
            ax.imshow(img[region], cmap="magma")
            ax.set_title(title)
            ax.axis("off")
    plt.tight_layout()
    plt.show()


def brightest_window(img: np.ndarray, size: int, step: int = 64) -> tuple[int, int]:
    # Top-left corner of the size x size window with the highest mean.
    best, pos = -np.inf, (0, 0)
    for y in range(0, img.shape[0] - size + 1, step):
        for x in range(0, img.shape[1] - size + 1, step):
            mean = img[y : y + size, x : x + size].mean()
            if mean > best:
                best, pos = mean, (y, x)
    return pos


frame = 0
CROP = 512
y0, x0 = brightest_window(gt[frame].sum(axis=-1), CROP)
show(frame)
show(frame, (slice(y0, y0 + CROP), slice(x0, x0 + CROP)))

Show the uncertainty for the same crop. High values show the pixels where the samples do not agree.

In [ ]:
region = (slice(y0, y0 + CROP), slice(x0, x0 + CROP))
fig, axes = plt.subplots(1, N_TARGETS, figsize=(6 * N_TARGETS, 5))
for ch, ax in enumerate(axes):
    im = ax.imshow(uncertainty[frame][region][..., ch], cmap="viridis")
    ax.set_title(f"Uncertainty ({MMSE_COUNT} samples): {names[ch]}")
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

### 6.4 Save the predictions

Save the predictions and the uncertainty as TIFF files with the axes `SYXC`.

In [ ]:
for name, arr in (("prediction", prediction), ("uncertainty", uncertainty)):
    out_path = WORK_DIR / f"{name}_{EVAL_SPLIT}_mmse{MMSE_COUNT}.tif"
    tifffile.imwrite(out_path, arr.astype(np.float32))
    print(f"Saved {out_path}")